# 🚗 Car Price Prediction with Machine Learning
### Internship Project — Task 3

---

**Objective:** Build a regression model to predict the selling price of used cars based on features like brand, mileage, fuel type, and transmission.

**Dataset:** 301 cars with 9 features  
**Target Variable:** `Selling_Price` (in Lakhs INR)  
**Libraries Used:** Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

---



---
## 1. Import Libraries <a id='1'></a>

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn — Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Scikit-learn — Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Scikit-learn — Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


---
## 2. Load & Explore Dataset <a id='2'></a>

In [2]:
data_path = next(Path.cwd().glob('**/car data.csv'), None)
if data_path is None:
    raise FileNotFoundError("Could not find 'car data.csv' in the current folder tree.")
df = pd.read_csv(data_path)

In [3]:
# Column data types
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())

--- Data Types ---
Car_Name             str
Year               int64
Selling_Price    float64
Present_Price    float64
Driven_kms         int64
Fuel_Type            str
Selling_type         str
Transmission         str
Owner              int64
dtype: object

--- Missing Values ---
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64


In [4]:
# Statistical summary
print('--- Statistical Summary ---')
df.describe().T.style.background_gradient(cmap='Blues')

--- Statistical Summary ---


,count,mean,std,min,25%,50%,75%,max
Year,301.000000,2013.627907,2.891554,2003.000000,2012.000000,2014.000000,2016.000000,2018.000000
Selling_Price,301.000000,4.661296,5.082812,0.100000,0.900000,3.600000,6.000000,35.000000
Present_Price,301.000000,7.628472,8.642584,0.320000,1.200000,6.400000,9.900000,92.600000
Driven_kms,301.000000,36947.205980,38886.883882,500.000000,15000.000000,32000.000000,48767.000000,500000.000000
Owner,301.000000,0.043189,0.247915,0.000000,0.000000,0.000000,0.000000,3.000000


In [5]:
# Unique values in categorical columns
categorical_cols = ['Fuel_Type', 'Selling_type', 'Transmission', 'Owner']
for col in categorical_cols:
    print(f'{col}: {df[col].unique()}')

Fuel_Type: <StringArray>
['Petrol', 'Diesel', 'CNG']
Length: 3, dtype: str
Selling_type: <StringArray>
['Dealer', 'Individual']
Length: 2, dtype: str
Transmission: <StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str
Owner: [0 1 3]


---
## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

### 3.1 Target Variable Distribution

In [6]:
fig1 = px.histogram(
    df,
    x='Selling_Price',
    nbins=30,
    title='Distribution of Selling Price',
    labels={'Selling_Price': 'Selling Price (Lakhs INR)', 'count': 'Count'}
)
fig1.add_vline(
    x=df['Selling_Price'].mean(),
    line_color='tomato',
    line_dash='dash',
    annotation_text=f"Mean: {df['Selling_Price'].mean():.2f}"
)
fig1.add_vline(
    x=df['Selling_Price'].median(),
    line_color='gold',
    line_dash='dash',
    annotation_text=f"Median: {df['Selling_Price'].median():.2f}"
)
fig1.update_layout(template='plotly_white')

fig2 = px.histogram(
    x=np.log1p(df['Selling_Price']),
    nbins=30,
    title='Log-Transformed Selling Price',
    labels={'x': 'log(1 + Selling Price)', 'count': 'Count'}
)
fig2.update_layout(template='plotly_white')

fig1.show()
fig2.show()

print(f'💡 Skewness of Selling_Price: {df["Selling_Price"].skew():.3f} (right-skewed — log transform will help)')

💡 Skewness of Selling_Price: 2.493 (right-skewed — log transform will help)


### 3.2 Categorical Feature Analysis

In [7]:
cat_features = ['Fuel_Type', 'Transmission', 'Selling_type']

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[f'Selling Price by {col}' for col in cat_features]
)

for i, col in enumerate(cat_features, start=1):
    order = df.groupby(col)['Selling_Price'].median().sort_values(ascending=False).index
    data = df.copy()
    data[col] = pd.Categorical(data[col], categories=order)
    fig.add_trace(
        go.Box(
            x=data[col],
            y=data['Selling_Price'],
            name=col,
            boxpoints='outliers',
            hovertemplate=f"{col}: %{{x}}<br>Selling Price: %{{y}} Lakhs<extra></extra>"
        ),
        row=1,
        col=i
    )

fig.update_layout(
    title='Price Distribution Across Categorical Features',
    template='plotly_white',
    showlegend=False,
    width=1400,
    height=500
)
fig.show()

### 3.3 Numerical Feature Analysis

In [8]:
numerical_cols = ['Year', 'Present_Price', 'Driven_kms', 'Owner']

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f'{col} vs Selling Price' for col in numerical_cols]
)

for i, col in enumerate(numerical_cols):
    row, col_idx = divmod(i, 2)
    z = np.polyfit(df[col], df['Selling_Price'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 200)

    fig.add_trace(
        go.Scatter(
            x=df[col],
            y=df['Selling_Price'],
            mode='markers',
            marker=dict(size=8, color='steelblue', opacity=0.6),
            name='Data',
            hovertemplate=f"{col}: %{{x}}<br>Selling Price: %{{y}} Lakhs<extra></extra>"
        ),
        row=row + 1,
        col=col_idx + 1
    )
    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=p(x_line),
            mode='lines',
            line=dict(color='tomato', dash='dash', width=2),
            name='Trend',
            showlegend=(i == 0),
            hovertemplate=f"Trend<{col}>: %{{y:.2f}}<extra></extra>"
        ),
        row=row + 1,
        col=col_idx + 1
    )

fig.update_layout(
    title='Numerical Features vs Selling Price',
    template='plotly_white',
    width=1300,
    height=900,
    showlegend=True
)
fig.show()

### 3.4 Correlation Heatmap

In [9]:
corr_df = df[['Year', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Owner']].corr()

fig = px.imshow(
    corr_df,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix (Numerical Features)'
)
fig.update_layout(template='plotly_white', width=800, height=700)
fig.show()

print('💡 Key Insight: Present_Price has the strongest positive correlation with Selling_Price')
print('💡 Key Insight: Driven_kms has a negative correlation — more driven = lower price')

💡 Key Insight: Present_Price has the strongest positive correlation with Selling_Price
💡 Key Insight: Driven_kms has a negative correlation — more driven = lower price


### 3.5 Top Car Brands by Average Selling Price

In [10]:
# Extract brand from Car_Name
df['Brand'] = df['Car_Name'].str.split().str[0].str.title()

brand_price = df.groupby('Brand')['Selling_Price'].mean().sort_values(ascending=False).head(15)

fig = px.bar(
    x=brand_price.values,
    y=brand_price.index,
    orientation='h',
    labels={'x': 'Average Selling Price (Lakhs INR)', 'y': 'Brand'},
    title='Top 15 Car Brands by Average Selling Price'
)
fig.update_layout(template='plotly_white', yaxis={'categoryarray': brand_price.index, 'categoryorder': 'array'})
fig.show()

---
## 4. Feature Engineering <a id='4'></a>

In [11]:
# Work on a copy
df_fe = df.copy()

# 1. Car Age (how old is the car)
CURRENT_YEAR = 2024
df_fe['Car_Age'] = CURRENT_YEAR - df_fe['Year']

# 2. Price Depreciation (how much value was lost compared to original)
df_fe['Price_Depreciation'] = df_fe['Present_Price'] - df_fe['Selling_Price']

# 3. Depreciation Rate (%)
df_fe['Depreciation_Rate'] = (df_fe['Price_Depreciation'] / df_fe['Present_Price']) * 100

# 4. Kms per Year (usage intensity)
df_fe['Kms_Per_Year'] = df_fe['Driven_kms'] / df_fe['Car_Age'].replace(0, 1)

# 5. Price per km driven (value density)
df_fe['Price_Per_Km'] = df_fe['Selling_Price'] / df_fe['Driven_kms'].replace(0, 1)

print('✅ New features created:')
new_features = ['Car_Age', 'Price_Depreciation', 'Depreciation_Rate', 'Kms_Per_Year', 'Price_Per_Km']
print(df_fe[new_features].head(5).to_string())
print(f'\n📊 New dataset shape: {df_fe.shape}')

✅ New features created:
   Car_Age  Price_Depreciation  Depreciation_Rate  Kms_Per_Year  Price_Per_Km
0       10                2.24          40.071556   2700.000000      0.000124
1       11                4.79          50.209644   3909.090909      0.000110
2        7                2.60          26.395939    985.714286      0.001051
3       13                1.30          31.325301    400.000000      0.000548
4       10                2.27          33.042213   4245.000000      0.000108

📊 New dataset shape: (301, 15)


In [12]:
# Visualize Car Age vs Selling Price
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=['Car Age vs Selling Price', 'Depreciation Rate vs Selling Price']
)

fig.add_trace(
    go.Scatter(
        x=df_fe['Car_Age'],
        y=df_fe['Selling_Price'],
        mode='markers',
        marker=dict(size=10, color='steelblue', opacity=0.6),
        hovertemplate='Car Age: %{x} years<br>Selling Price: %{y} Lakhs<extra></extra>'
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=df_fe['Depreciation_Rate'],
        y=df_fe['Selling_Price'],
        mode='markers',
        marker=dict(size=10, color='darkorange', opacity=0.6),
        hovertemplate='Depreciation Rate: %{x}%<br>Selling Price: %{y} Lakhs<extra></extra>'
    ),
    row=1,
    col=2
)
fig.update_layout(
    title='Engineered Features Analysis',
    template='plotly_white',
    width=1300,
    height=500
)
fig.show()

---
## 5. Data Preprocessing <a id='5'></a>

In [13]:
df_model = df_fe.copy()

# ── Label Encoding for categorical features ──
le = LabelEncoder()
categorical_cols = ['Fuel_Type', 'Selling_type', 'Transmission']

encoding_map = {}
for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col])
    encoding_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'{col}: {encoding_map[col]}')

# ── Drop columns not needed for modeling ──
drop_cols = ['Car_Name', 'Brand', 'Year']   # Year replaced by Car_Age; Car_Name/Brand too granular
df_model.drop(columns=drop_cols, inplace=True)

print(f'\n✅ Encoding done. Remaining columns: {list(df_model.columns)}')

Fuel_Type: {'CNG': np.int64(0), 'Diesel': np.int64(1), 'Petrol': np.int64(2)}
Selling_type: {'Dealer': np.int64(0), 'Individual': np.int64(1)}
Transmission: {'Automatic': np.int64(0), 'Manual': np.int64(1)}

✅ Encoding done. Remaining columns: ['Selling_Price', 'Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Car_Age', 'Price_Depreciation', 'Depreciation_Rate', 'Kms_Per_Year', 'Price_Per_Km']


In [14]:
# ── Define Features (X) and Target (y) ──
X = df_model.drop(columns=['Selling_Price'])
y = df_model['Selling_Price']

print(f'Features (X): {list(X.columns)}')
print(f'Target (y): Selling_Price')
print(f'X shape: {X.shape} | y shape: {y.shape}')

Features (X): ['Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Car_Age', 'Price_Depreciation', 'Depreciation_Rate', 'Kms_Per_Year', 'Price_Per_Km']
Target (y): Selling_Price
X shape: (301, 11) | y shape: (301,)


In [15]:
# ── Train-Test Split (80:20) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Feature Scaling ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set  : {X_train.shape[0]} samples')
print(f'Testing set   : {X_test.shape[0]} samples')
print('✅ Scaling done with StandardScaler')

Training set  : 240 samples
Testing set   : 61 samples
✅ Scaling done with StandardScaler


---
## 6. Model Training <a id='6'></a>

We train **6 models** and compare them to find the best one.

In [16]:
# ── Define all models ──
models = {
    'Linear Regression'       : LinearRegression(),
    'Ridge Regression'        : Ridge(alpha=1.0),
    'Lasso Regression'        : Lasso(alpha=0.1),
    'Decision Tree'           : DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest'           : RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    'Gradient Boosting'       : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
}

results = []

for name, model in models.items():
    # Use scaled data for linear models, raw for tree-based
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    cv_r2 = cv_scores.mean()

    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R² Score': r2, 'CV R² (5-fold)': cv_r2})
    print(f'{name:<25} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.4f} | CV R²={cv_r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False).reset_index(drop=True)
print('\n✅ All models trained!')

Linear Regression         | MAE=0.000 | RMSE=0.000 | R²=1.0000 | CV R²=1.0000
Ridge Regression          | MAE=0.157 | RMSE=0.272 | R²=0.9968 | CV R²=0.9970
Lasso Regression          | MAE=0.383 | RMSE=0.618 | R²=0.9834 | CV R²=0.9845
Decision Tree             | MAE=0.595 | RMSE=0.910 | R²=0.9641 | CV R²=0.8278
Random Forest             | MAE=0.420 | RMSE=0.709 | R²=0.9782 | CV R²=0.8778
Gradient Boosting         | MAE=0.303 | RMSE=0.546 | R²=0.9870 | CV R²=0.8874

✅ All models trained!


---
## 7. Model Evaluation & Comparison <a id='7'></a>

In [17]:
# Results table
print('\n📊 Model Performance Summary (sorted by R² Score):')
results_df.style \
    .background_gradient(subset=['R² Score', 'CV R² (5-fold)'], cmap='Greens') \
    .background_gradient(subset=['MAE', 'RMSE'], cmap='Reds_r') \
    .format({'MAE': '{:.3f}', 'RMSE': '{:.3f}', 'R² Score': '{:.4f}', 'CV R² (5-fold)': '{:.4f}'})


📊 Model Performance Summary (sorted by R² Score):


,Model,MAE,RMSE,R² Score,CV R² (5-fold)
0,Linear Regression,0.000,0.000,1.0000,1.0000
1,Ridge Regression,0.157,0.272,0.9968,0.9970
2,Gradient Boosting,0.303,0.546,0.9870,0.8874
3,Lasso Regression,0.383,0.618,0.9834,0.9845
4,Random Forest,0.420,0.709,0.9782,0.8778
5,Decision Tree,0.595,0.910,0.9641,0.8278


In [18]:
# Comparison bar chart
metrics = ['R² Score', 'MAE', 'RMSE']
colors_list = ['#2ecc71', '#e74c3c', '#e67e22']

fig = make_subplots(rows=1, cols=3, subplot_titles=[f'Model Comparison — {metric}' for metric in metrics])

for i, (metric, color) in enumerate(zip(metrics, colors_list), start=1):
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'R² Score'))
    fig.add_trace(
        go.Bar(
            x=sorted_df[metric],
            y=sorted_df['Model'],
            orientation='h',
            marker=dict(color=color),
            text=[f'{v:.3f}' for v in sorted_df[metric]],
            textposition='outside',
            hovertemplate=f"{metric}: %{{x:.4f}}<extra></extra>"
        ),
        row=1,
        col=i
    )

fig.update_layout(
    title='Model Performance Comparison',
    template='plotly_white',
    width=1700,
    height=500,
    showlegend=False
)
fig.show()

In [19]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[f'{best_model_name} Actual vs Predicted', f'{best_model_name} Residual Plot'])

# Actual vs Predicted
min_val, max_val = min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())
fig.add_trace(
    go.Scatter(
        x=y_test,
        y=y_pred_best,
        mode='markers',
        marker=dict(size=9, color='steelblue', opacity=0.7),
        name='Predictions',
        hovertemplate='Actual: %{x}<br>Predicted: %{y}<extra></extra>'
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Perfect Prediction'
    ),
    row=1,
    col=1
)

# Residuals
residuals = y_test.values - y_pred_best
fig.add_trace(
    go.Scatter(
        x=y_pred_best,
        y=residuals,
        mode='markers',
        marker=dict(size=9, color='darkorange', opacity=0.7),
        name='Residuals',
        hovertemplate='Predicted: %{x}<br>Residual: %{y}<extra></extra>'
    ),
    row=1,
    col=2
)
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[0, 0],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Zero Residual'
    ),
    row=1,
    col=2
)

fig.update_layout(
    title=f'Best Model Analysis: {best_model_name}',
    template='plotly_white',
    width=1400,
    height=500,
    showlegend=True
)
fig.show()

print(f'🏆 Best Model: {best_model_name}')
print(f'   R² Score : {results_df.iloc[0]["R² Score"]:.4f}')
print(f'   MAE      : {results_df.iloc[0]["MAE"]:.4f} Lakhs')
print(f'   RMSE     : {results_df.iloc[0]["RMSE"]:.4f} Lakhs')

NameError: name 'best_model_name' is not defined

---
## 8. Feature Importance <a id='8'></a>

In [ ]:
fig = px.bar(
    feature_importance,
    x='Importance',
    y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale='RdYlGn',
    title='Feature Importance (Random Forest)'
)
fig.update_layout(
    template='plotly_white',
    xaxis_title='Importance Score',
    yaxis_title='Feature'
)
fig.update_yaxes(categoryarray=feature_importance['Feature'][::-1], categoryorder='array')
fig.show()

NameError: name 'feature_importance' is not defined

---
## 9. Predict on New Data <a id='9'></a>

Simulate a real-world prediction on an unseen car.

In [ ]:
# ── Example: Predict price of a new car ──
# Encoding reference:
#   Fuel_Type   : CNG=0, Diesel=1, Petrol=2
#   Selling_type: Dealer=0, Individual=1
#   Transmission: Automatic=0, Manual=1

new_car = {
    'Present_Price'     : 8.5,      # Showroom price (Lakhs)
    'Driven_kms'        : 45000,    # Total kms driven
    'Fuel_Type'         : 2,        # Petrol
    'Selling_type'      : 0,        # Dealer
    'Transmission'      : 1,        # Manual
    'Owner'             : 0,        # First owner
    'Car_Age'           : 5,        # 5 years old (2024 - 2019)
    'Price_Depreciation': 8.5 - 5,  # Rough estimate
    'Depreciation_Rate' : ((8.5 - 5) / 8.5) * 100,
    'Kms_Per_Year'      : 45000 / 5,
    'Price_Per_Km'      : 5 / 45000
}

new_car_df = pd.DataFrame([new_car])
print('🚗 New Car Details:')
print(new_car_df.to_string(index=False))

# Predict with all models
print('\n💰 Price Predictions:')
for name, model in models.items():
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        new_scaled = scaler.transform(new_car_df)
        pred = model.predict(new_scaled)[0]
    else:
        pred = model.predict(new_car_df)[0]
    print(f'  {name:<25}: ₹ {pred:.2f} Lakhs')

In [ ]:
fig = px.bar(
    preds_df,
    x='Model',
    y='Predicted Price (Lakhs)',
    color='Model',
    color_discrete_sequence=['#27ae60' if model == best_model_name else '#3498db' for model in preds_df['Model']],
    text=[f'₹{v:.2f}L' for v in preds_df['Predicted Price (Lakhs)']],
    title='New Car Price Predictions Across All Models'
)
fig.update_layout(
    template='plotly_white',
    xaxis_title='Model',
    yaxis_title='Predicted Price (Lakhs INR)',
    legend_title='Legend'
)
fig.update_traces(textposition='outside')
fig.show()

NameError: name 'preds_df' is not defined

---
## 10. Conclusion <a id='10'></a>

In [ ]:
best_row = results_df.iloc[0]

print('=' * 60)
print('         CAR PRICE PREDICTION — PROJECT SUMMARY')
print('=' * 60)
print(f"\n📦 Dataset       : 301 used cars with 9 features")
print(f"🔧 Features Used : {X.shape[1]} (including 5 engineered)")
print(f"\n🏆 Best Model    : {best_row['Model']}")
print(f"   R² Score     : {best_row['R² Score']:.4f}  → explains {best_row['R² Score']*100:.1f}% of price variance")
print(f"   MAE          : {best_row['MAE']:.4f} Lakhs → avg error of ₹{best_row['MAE']*100:.0f}K")
print(f"   RMSE         : {best_row['RMSE']:.4f} Lakhs")

print("\n📌 Key Findings:")
print("   1. Present_Price is the strongest predictor of Selling_Price")
print("   2. Car Age negatively impacts price — older cars sell for less")
print("   3. Diesel cars have a higher median resale value than Petrol")
print("   4. Automatic transmission commands a price premium")
print("   5. Higher driven kms → lower resale value")
print("   6. Engineered features (Car_Age, Depreciation_Rate) improved model accuracy")

print("\n🔮 Possible Improvements:")
print("   • Hyperparameter tuning with GridSearchCV / RandomizedSearchCV")
print("   • Try XGBoost / LightGBM for potentially higher accuracy")
print("   • Collect more data (luxury brands, electric vehicles)")
print("   • Add geographic features (city-wise demand)")
print('=' * 60)